# Example 10.4.3 / 3: Approximate a Concave Profit Function

Companion notebook for **10.4.3 Automatic Piecewise-Linear Approximation** in the AMPL lab Documentation, section "Approximate a Concave Profit Function". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Approximate a Concave Profit Function

A plant's revenue grows with diminishing returns, $40\log(1 + m)$, and each unit costs 5:

$$
\max_{0 \le m \le 10}\; 40\log(1+m) - 5m .
$$

The exact optimum satisfies $40/(1+m) = 5$, so $m^* = 7$ and the profit is
$40\log 8 - 35 \approx 48.1777$. HiGHS solves an approximation:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param unit_cost >= 0;  # cost per unit made
var Make >= 0, <= 10;  # units made

# revenue with diminishing returns, minus a linear production cost
maximize Profit: 40 * log(1 + Make) - unit_cost * Make;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["unit_cost"] = 5

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(ampl.var["Make"].value(), ampl.obj["Profit"].value())

solved
6.913369027431431 48.17529930806811


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`. `option solver_msg 0` drops HiGHS's closing summary, so
the approximation warning is printed once:

In [6]:
%%writefile data.dat
param unit_cost := 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display Make, Profit;

HiGHS 1.14.0: 
------------ WARNINGS ------------
  An expression of type 'Log' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.010000).
Make = 6.91337
Profit = 48.1753



### Another way in amplpy

The `PLApprox_Log` warning names the approximated function and the current tolerance. The returned
`Make` is close to 7 but not equal to it. `ampl.obj["Profit"].value()` and `display Profit` are
computed by AMPL from the original `log` expression at the returned `Make`, so they are the true
profit of the returned decision, which is slightly below the exact optimum.

**Control the relative tolerance.** Option `cvt:plapprox:reltol` (synonyms `plapprox:reltol`,
`plapproxreltol`) sets the relative tolerance of the approximation; its default is 0.01. A
smaller value asks for a closer approximation, which needs more pieces; a larger value gives a
coarser one. With the same model and data, solved at three tolerances:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["unit_cost"] = 5

In [11]:
# run.py
for tol in [0.1, 0.01, 0.0001]:
    ampl.solve(solver="highs", verbose=False,
               highs_options=f"cvt:plapprox:reltol={tol}")
    print(tol, ampl.solve_result,
          round(ampl.var["Make"].value(), 4), round(ampl.obj["Profit"].value(), 4))

0.1 solved 5.8092 47.685
0.01 solved 6.9134 48.1753
0.0001 solved 6.9841 48.1776


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`. `solve > highs.log` sends the solver's remaining messages
to a file:

In [12]:
%%writefile data_2.dat
param unit_cost := 5;

Writing data_2.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data_2.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
for {tol in {0.1, 0.01, 0.0001}} {
    option highs_options ('cvt:plapprox:reltol=' & tol);
    solve > highs.log;
    printf "%g %s %.4f %.4f\n", tol, solve_result, Make, Profit;
}

0.1 solved 5.8092 47.6850
0.01 solved 6.9134 48.1753
0.0001 solved 6.9841 48.1776
